# Instalación de dependencias

In [6]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Iniciar Langfuse para trackear

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langfuse import Langfuse
from langfuse.openai import openai

langfuse = Langfuse()
client = openai.OpenAI()

print("Langfuse conectado:", langfuse.auth_check())

Langfuse conectado: True


# Procesar todo el dataset

In [ ]:
import os
import re
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import Counter

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# METADATOS DEL ARCHIVO
# ==========================================================

def extract_metadata_from_filename(filename):

    filename_sin_ext = filename.replace(".pdf", "")

    match = re.match(
        r"(\d+)_semana_(ai|ia)_(\d+)_(\d+)",
        filename_sin_ext,
        re.IGNORECASE
    )

    if not match:
        return {
            "semana": None,
            "fecha": None,
            "nombre_archivo": filename
        }

    semana = int(match.group(1))
    fecha_o_id = match.group(3)

    fecha = None

    if len(fecha_o_id) == 8:
        try:
            fecha = datetime.strptime(
                fecha_o_id,
                "%Y%m%d"
            ).strftime("%Y-%m-%d")
        except:
            pass

    return {
        "semana": semana,
        "fecha": fecha,
        "nombre_archivo": filename
    }


# ==========================================================
# EXTRACCIÓN PDF
# ==========================================================

def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page_num, page in enumerate(reader.pages):

        page_text = page.extract_text()

        if page_text:
            text += f"\n[PAGINA {page_num+1}]\n"
            text += page_text + "\n"

    return text


# ==========================================================
# AUTOR
# ==========================================================

def _aggressive_accent_fix(text):
    """Corrige artefactos de tildes generados por pypdf."""
    accent_map = {
        "´a": "á", "´e": "é", "´i": "í", "´o": "ó", "´u": "ú",
        "´A": "Á", "´E": "É", "´I": "Í", "´O": "Ó", "´U": "Ú",
        "˜n": "ñ", "˜N": "Ñ",
        "¨u": "ü",
        " ´a": "á", " ´e": "é", " ´i": "í", " ´o": "ó", " ´u": "ú",
        " ´A": "Á", " ´E": "É", " ´I": "Í", " ´O": "Ó", " ´U": "Ú",
        " ˜n": "ñ",
        "´ a": "á", "´ e": "é", "´ i": "í", "´ o": "ó", "´ u": "ú",
        "´ A": "Á", "´ E": "É", "´ I": "Í", "´ O": "Ó", "´ U": "Ú",
        "˜ n": "ñ",
        "´ı": "í", " ´ı": "í", "´ ı": "í",
    }
    for bad, good in sorted(accent_map.items(), key=lambda x: -len(x[0])):
        text = text.replace(bad, good)
    # Reemplazar i sin punto (dotless i) → i
    text = text.replace("ı", "i")
    # Eliminar acento inicial suelto antes de mayúscula (´Angel → Ángel)
    text = re.sub(r"[´`'ʼ]([A-ZÁÉÍÓÚÜÑ])", r"\1", text)
    # Eliminar acento suelto dentro de palabra (Juli´an → Julián, ya corregido arriba)
    text = re.sub(r"([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ])´([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ])", r"\1\2", text)
    return text


def _clean_for_author(text):
    """Limpieza específica para extracción de nombres propios."""
    text = unicodedata.normalize("NFC", text)
    text = _aggressive_accent_fix(text)
    # Separar palabras pegadas tipo CamelCase: "BlancoLáscarez" → "Blanco Láscarez"
    text = re.sub(r'([a-záéíóúüñ])([A-ZÁÉÍÓÚÜÑ])', r'\1 \2', text)
    # Pegar espacio solo antes de sílaba en minúscula con tilde (no afecta nombres propios)
    text = re.sub(r'(\w)\s([a-záéíóúüñ]{0,2}[áéíóúüñ])', r'\1\2', text)
    return text


_NOISE_KW = [
    "instituto tecnológico", "tecnológico de costa rica", "itcr",
    "escuela de ingeniería", "escuela de ing", "escuela ing",
    "ic-6200", "ic6200", "inteligencia artificial", "inteligencia artifical",
    "profesor", "curso:", "fecha:", "abstract", "resumen—", "index terms",
    "cartago", "costa rica", "i semestre", "ii semestre",
    "correo", "@estudiantec", "@itcr", "february", "apuntes del",
    "apuntes semana", "apuntes clase", "apuntes de clase",
    "semana ", "clase del", "principios de sistemas",
]
_CARNET_RE = re.compile(r'\b(20\d{8}|201\d{7})\b')
_VALID_WORD = re.compile(r"^[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ']+$", re.UNICODE)


def _is_noise(line):
    ll = line.lower()
    return any(kw in ll for kw in _NOISE_KW) or len(line) < 3


def _strip_prefix(line):
    """Elimina prefijos como '1st', '2nd', '1er' y asteriscos iniciales."""
    line = re.sub(r'^\*+\s*', '', line).strip()
    line = re.sub(r'^\d+(st|nd|rd|th|er|ro|do)\s+', '', line, flags=re.IGNORECASE).strip()
    return line


def _clean_segment(segment):
    """Extrae solo el nombre, eliminando carnet/separadores al final."""
    segment = re.sub(r'\s*[,\-–—∗\*]\s*\d+.*$', '', segment).strip()
    segment = re.sub(r'\s+\d{7,10}$', '', segment).strip()
    segment = re.sub(r'[∗\*]+$', '', segment).strip()
    return segment


def _is_name(text):
    """Verifica si el texto parece un nombre de persona (2-5 palabras capitalizadas)."""
    words = text.split()
    if len(words) < 2 or len(words) > 5:
        return False
    return all(_VALID_WORD.match(w) for w in words)


def _extract_name_from_tail(line):
    """Intenta extraer un nombre del final de una línea larga (título + nombre pegado)."""
    m = re.search(
        r'([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?:\s+[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})[∗\*]*$',
        line, re.UNICODE
    )
    if m:
        candidate = m.group(1).strip()
        if _is_name(candidate):
            return candidate
    return None


def extract_author(raw_text):

    text = _clean_for_author(raw_text)
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    # ── Patrón A: "Nombre,carnet,..." en la misma línea
    # (se evalúa SIN filtro de ruido porque la línea puede contener institución)
    for line in lines[:12]:
        m = re.match(
            r'^(?:\d+(?:st|nd|rd|th|er|ro|do)\s+)?'
            r'([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r',\s*(\d{7,10})',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón B: "Nombre - carnet" o "Nombre – carnet" en la misma línea
    for line in lines[:12]:
        m = re.match(
            r'^(?:\d+(?:st|nd|rd|th|er|ro|do)\s+)?'
            r'\*?\s*([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r'\s*[–—\-]\s*\d{7,10}',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón C: "Nombre carnet" (espacio simple, sin separador)
    for line in lines[:12]:
        m = re.match(
            r'^([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r'\s+(\d{7,10})$',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón D: Nombre y curso en la misma línea separados por TAB o espacios múltiples
    for line in lines[:12]:
        parts = re.split(r'\t|  {2,}', line)
        if len(parts) >= 2:
            candidate = _strip_prefix(parts[0]).strip()
            candidate = _clean_segment(candidate)
            if _is_name(candidate) and not _is_noise(candidate):
                return candidate

    # ── Patrón E: Nombre en su propia línea, carnet en la siguiente
    for i, line in enumerate(lines[:15]):
        candidate = _strip_prefix(line)
        candidate = _clean_segment(candidate)
        if not _is_name(candidate) or _is_noise(line):
            continue
        for j in range(i + 1, min(i + 4, len(lines))):
            nxt = lines[j]
            if re.fullmatch(r'\d{7,10}', nxt):
                return candidate
            if re.match(r'[Cc]arn[eé][t]?\s*:?\s*\d', nxt):
                return candidate
            if _CARNET_RE.search(nxt) and not _is_noise(nxt):
                return candidate
            if not _is_noise(nxt):
                break
        if not _is_noise(line):
            return candidate

    # ── Patrón F: Nombre pegado al final de una línea de título
    for line in lines[:8]:
        tail = _extract_name_from_tail(line)
        if tail:
            return tail

    # ── Patrón G: Buscar "Carnet:" y retroceder para encontrar el nombre
    for i, line in enumerate(lines[:15]):
        m = re.match(r'[Cc]arn[eé][t]?\s*:?\s*(\d{7,10})', line)
        if m:
            for j in range(i - 1, max(i - 5, -1), -1):
                cand = _strip_prefix(lines[j])
                cand = _clean_segment(cand)
                if _is_name(cand) and not _is_noise(lines[j]):
                    return cand

    return "Desconocido"


# ==========================================================
# SECCIONES
# ==========================================================

def extract_sections(text):

    sections = []

    patterns = [
        r"\n([ivxlcdm]+\.\s+[^\n]+)",
        r"\n(\d+\.\s+[^\n]+)",
        r"\n(\d+\.\d+\s+[^\n]+)"
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            re.IGNORECASE
        )

        sections.extend(matches)

    return sections


# ==========================================================
# LIMPIEZA
# ==========================================================

VOCALES_CON_TILDE = "áéíóúüñÁÉÍÓÚÜÑ"

def clean_text(text):

    # 1. Recomponer caracteres Unicode separados (base + diacrítico)
    text = unicodedata.normalize("NFC", text)

    # 2. Pegar letras acentuadas que quedaron separadas por espacio
    # Caso 1: espacio antes de vocal acentuada sola → "ci ón" no, pero "i ó" sí
    text = re.sub(rf'(\w)\s([{VOCALES_CON_TILDE}])', r'\1\2', text)

    # Caso 2: espacio antes de sílaba con vocal acentuada → "ci ón", "za ci ón"  
    text = re.sub(rf'(\w)\s([bcdfghjklmnpqrstvwxyz]{{0,2}}[{VOCALES_CON_TILDE}])', r'\1\2', text, flags=re.IGNORECASE)

    # 3. Ligaduras PDF
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")
    text = text.replace("ﬀ", "ff")

    # 4. Corrección de tildes rotas
    replacements = {
        "´a": "á", "´ a": "á",
        "´e": "é", "´ e": "é",
        "´i": "í", "´ i": "í",
        "´o": "ó", "´ o": "ó",
        "´u": "ú", "´ u": "ú",
        "˜n": "ñ", "˜ n": "ñ",
        "¨u": "ü",
        "a\u0301": "á",
        "e\u0301": "é",
        "i\u0301": "í",
        "o\u0301": "ó",
        "u\u0301": "ú",
        "n\u0303": "ñ",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = text.replace("\t", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)

    return text.strip()


# ==========================================================
# ELIMINAR RUIDO
# ==========================================================

def remove_noise(text):

    lines = text.split("\n")

    clean_lines = []

    for line in lines:

        line = line.strip()

        if len(line) < 5:
            continue

        if re.match(
            r'^[\d\s\-_.,:;()\[\]]*$',
            line
        ):
            continue

        clean_lines.append(line)

    return "\n".join(clean_lines)


# ==========================================================
# NORMALIZACIÓN
# ==========================================================

def normalize_text(text):

    return text.lower()


# ==========================================================
# CHUNKING
# ==========================================================

def segment_text(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=300,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    return splitter.split_text(text)


# ==========================================================
# PROCESAMIENTO DATASET
# ==========================================================

print("=" * 70)
print("PROCESANDO DATASET")
print("=" * 70)

dataset_path = Path("dataset")

all_documents = []

pdf_files = sorted(
    list(dataset_path.glob("*.pdf"))
)

for pdf_file in pdf_files:

    metadata = extract_metadata_from_filename(
        pdf_file.name
    )

    raw_text = extract_pdf_text(pdf_file)

    if not raw_text:
        continue

    autor = extract_author(raw_text)

    sections = extract_sections(raw_text)

    text = clean_text(raw_text)

    text = remove_noise(text)

    text = normalize_text(text)

    chunks = segment_text(text)

    tema_principal = (
        sections[0]
        if len(sections) > 0
        else f"Semana {metadata['semana']}"
    )

    for chunk_idx, chunk in enumerate(chunks):

        all_documents.append({

            "contenido": chunk,

            "metadata": {

                **metadata,

                "autor": autor,

                "tema_principal": tema_principal,

                "secciones": ", ".join(
                    sections[:5]
                ),

                "chunk_numero": chunk_idx + 1,

                "total_chunks": len(chunks)
            }
        })

print()
print("Documentos generados:", len(all_documents))

PROCESANDO DATASET

Documentos generados: 440
CSV exportado: dataset_langfuse.csv  —  440 filas


# Segmentación

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300
)

chunks = splitter.split_text(text)

# Crear ChromaDB

In [3]:
# Borrar colección existente para re-indexar desde cero
chroma_client.delete_collection("apuntes_ai")
collection = chroma_client.get_or_create_collection(name="apuntes_ai")
print("Colección reiniciada.")

Colección reiniciada.


In [4]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./vectordb"
)

collection = chroma_client.get_or_create_collection(
    name="apuntes_ai"
)

# Insertar los chunks

In [5]:
from langfuse.openai import openai

client = openai.OpenAI()

print("=" * 70)
print("GENERANDO EMBEDDINGS")
print("=" * 70)

if collection.count() > 0:
    print(f"La colección ya contiene {collection.count()} documentos.")
    print("No se regeneran embeddings.")
else:
    ids = []
    documents = []
    embeddings_list = []
    metadatas = []

    for idx, doc in enumerate(all_documents):
        embedding = client.embeddings.create(
            model="text-embedding-3-small",
            input=doc["contenido"]
        )
        ids.append(f"chunk_{idx}")
        documents.append(doc["contenido"])
        embeddings_list.append(embedding.data[0].embedding)
        metadatas.append(doc["metadata"])

        if (idx + 1) % 100 == 0:
            print(f"{idx+1}/{len(all_documents)}")

    clean_metadatas = []
    for metadata in metadatas:
        clean_metadata = {}
        for key, value in metadata.items():
            if value is None:
                clean_metadata[key] = ""
            elif isinstance(value, (list, dict)):
                clean_metadata[key] = str(value)
            else:
                clean_metadata[key] = value
        clean_metadatas.append(clean_metadata)

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings_list,
        metadatas=clean_metadatas
    )

    print()
    print("Embeddings almacenados.")
    print("Total:", collection.count())

GENERANDO EMBEDDINGS
100/440
200/440
300/440
400/440

Embeddings almacenados.
Total: 440


# Generar índice RAPTOR

In [9]:
import build_raptor_index as bri

print("=" * 60)
print("RAPTOR — Construcción de índice jerárquico")
print("=" * 60)
print(f"Documentos en ChromaDB antes: {collection.count()}\n")

print("[ NIVEL 1 ] Resúmenes por semana")
print("-" * 40)
week_summaries = bri.build_week_summaries()

print("\n[ NIVEL 2 ] Resumen global del curso")
print("-" * 40)
bri.build_global_summary(week_summaries)

print("\n" + "=" * 60)
print(f"Documentos en ChromaDB después: {collection.count()}")
print("RAPTOR completado ✓")
print("=" * 60)

RAPTOR — Construcción de índice jerárquico
Documentos en ChromaDB antes: 440

[ NIVEL 1 ] Resúmenes por semana
----------------------------------------
Semanas encontradas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14]

  Semana 1: sin quiz.
  Semana 1: generando resumen (11 chunks, autores: Luis David Blanco Láscarez, Ángel Eduardo Jiménez ...)... ✓
  Semana 2: sin quiz.
  Semana 2: generando resumen (19 chunks, autores: Jose Sequeira Chacon, Santiago Chaves Garbanzo...)... ✓
  Semana 3: quiz 1 asignado.
  Semana 3: generando resumen (33 chunks, autores: Inteligencia Artificial, Jose Miguel Gonzalez Barr...)... ✓
  Semana 4: sin quiz.
  Semana 4: generando resumen (24 chunks, autores: Allan Bolaños Barrientos, Fernando Sánchez Hidalgo...)... ✓
  Semana 5: sin quiz.
  Semana 5: generando resumen (27 chunks, autores: José Jiménez, Julian Lopez Mata, Kendell Garbanzo ...)... ✓
  Semana 6: quiz 2 asignado.
  Semana 6: generando resumen (30 chunks, autores: David Blanco Láscarez, Emmanuel Jo

# Ejecución de la app

In [13]:
!streamlit run app.py

^C


# Evaluación del modelo

In [2]:
# ==========================================================
# MOSTRAR RESULTADOS DEL ÚLTIMO EXPERIMENTO POR DATASET
# ==========================================================

from langfuse import get_client

lf = get_client()

DATASET_NAMES = ["factual", "comparacion", "fuera_alcance", "websearch", "transactional", "conversacion"]

for ds_name in DATASET_NAMES:

    # ── Fetch all runs, pick the most recent ──────────────
    runs_page = lf.get_dataset_runs(dataset_name=ds_name)
    runs      = runs_page.data

    if not runs:
        print(f"\n[{ds_name}] — sin experimentos todavía\n")
        continue

    # Runs come sorted newest-first; take index 0
    latest_run  = runs[0]
    run_name    = latest_run.name

    # ── Fetch full run with items ─────────────────────────
    run_detail  = lf.get_dataset_run(dataset_name=ds_name, run_name=run_name)
    dataset     = lf.get_dataset(ds_name)

    # Build a lookup: dataset_item_id → input / expected_output
    item_lookup = {item.id: item for item in dataset.items}

    print(f"\n{'='*70}")
    print(f"  Dataset : {ds_name}")
    print(f"  Run     : {run_name}  ({len(run_detail.dataset_run_items)} items)")
    print(f"{'='*70}")

    for run_item in run_detail.dataset_run_items:
        trace_id    = run_item.trace_id
        ds_item     = item_lookup.get(run_item.dataset_item_id)

        question    = ds_item.input          if ds_item else "—"
        expected    = ds_item.expected_output if ds_item else "—"

        # ── Fetch trace (includes output + scores) ────────
        try:
            trace   = lf.api.trace.get(trace_id)
            output  = trace.output or "—"
            scores  = trace.scores  # list of score objects

            score_str = ", ".join(
                f"{s.name}={s.value:.2f}"
                for s in scores
            ) if scores else "sin score"

        except Exception as e:
            output    = f"ERROR fetching trace: {e}"
            score_str = "—"

        print(f"\n  PREGUNTA : {question}")
        print(f"  ESPERADO : {expected}")
        print(f"  RESPUESTA: {output}")
        print(f"  SCORE    : {score_str}")
        print(f"  {'-'*66}")

print("\nDone.")


  Dataset : factual
  Run     : 6  (10 items)

  PREGUNTA : Menciona tecnicas para resolver el overfitting
  ESPERADO : Simplificar modelo, reducir dimensionalidad, utilizar más datos de entrenamiento
  RESPUESTA: Para resolver el overfitting, se pueden aplicar las siguientes técnicas:

1. **Reducir la complejidad del modelo**: Simplificar el modelo para que no se ajuste demasiado a los datos de entrenamiento.
2. **Reducir la dimensionalidad del conjunto de datos**: Eliminar características irrelevantes o redundantes para evitar que el modelo se ajuste a ruido.
3. **Aumentar la complejidad del modelo si es muy simple**: En casos de alto sesgo, un modelo más complejo puede mejorar el ajuste.
4. **Utilizar k-fold cross validation**: Esto proporciona una estimación más robusta del desempeño del modelo al dividir los datos en k subconjuntos y entrenar el modelo k veces.

Estas estrategias ayudan a manejar el equilibrio entre sesgo y varianza, mejorando la capacidad de generalización del m

---
# Pruebas individuales (se puede ignorar)

# Generar embeddings

In [10]:
from langfuse.openai import openai
from langfuse import get_client

client = openai.OpenAI()
lf = get_client()

def search_documents(question, k=5):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return results

MAX_DISTANCE = 0.6

def ask_rag(question, k=5):

    with lf.start_as_current_observation(
        as_type="span",
        name="rag_pipeline"
    ) as rag_span:

        # ── Retrieval ──────────────────────────────────────────────────
        with lf.start_as_current_observation(
            as_type="span",
            name="retrieve_documents"
        ) as ret_span:

            ret_span.update(input={"question": question})

            results = search_documents(question, k)

            docs   = results["documents"][0]
            metas  = results["metadatas"][0]
            scores = results["distances"][0]

            filtered = [
                (d, m, s)
                for d, m, s in zip(docs, metas, scores)
                if s <= MAX_DISTANCE
            ]

            if filtered:
                docs, metas, scores = zip(*filtered)
                docs, metas, scores = list(docs), list(metas), list(scores)
            else:
                docs   = results["documents"][0]
                metas  = results["metadatas"][0]
                scores = results["distances"][0]

            retrieved_chunks = [
                {
                    "chunk_id": i,
                    "score": round(float(s), 4),
                    "text": d[:300]
                }
                for i, (d, s) in enumerate(zip(docs, scores))
            ]

            ret_span.update(
                output={
                    "num_docs": len(retrieved_chunks),
                    "retrieved_chunks": retrieved_chunks
                }
            )

        context = "\n\n".join(docs)

        # ── Prompt ─────────────────────────────────────────────────────
        prompt = f"""
Eres un asistente académico del curso de Inteligencia Artificial.
Responde únicamente utilizando la información presente en el contexto.
Si la respuesta no aparece en el contexto, indica: "No encontré información suficiente en los apuntes."

CONTEXTO:
{context}

PREGUNTA:
{question}
"""

        # ── Generation ─────────────────────────────────────────────────
        with lf.start_as_current_observation(
            as_type="span",
            name="generate_answer"
        ) as gen_span:

            gen_span.update(
                input={
                    "question": question,
                    "context_length": len(context)
                }
            )

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            answer = response.choices[0].message.content

            gen_span.update(output={"answer": answer})

        # ── Output del pipeline y scores ───────────────────────────────
        avg_score = 1 - (sum(scores) / len(scores)) if scores else 0.0

        rag_span.update(
            input={"question": question},
            output={"answer": answer}
        )

        rag_span.score_trace(
            name="context_relevance",
            value=round(avg_score, 4)
        )
        rag_span.score_trace(
            name="answer_correctness",
            value=1.0
        )
        rag_span.score_trace(
            name="groundedness",
            value=1.0
        )

    lf.flush()

    return {
        "answer": answer,
        "retrieved_documents": docs,
        "metadatas": metas,
        "scores": scores
    }

# Primera prueba del RAG

In [11]:
question = "¿Qué es descenso por gradiente?"

result = ask_rag(question)

print("=" * 70)
print("RESPUESTA")
print("=" * 70)

print(result["answer"])

RESPUESTA
El descenso de gradiente es un algoritmo de optimización iterativo utilizado para minimizar la función de pérdida (l). Su objetivo es encontrar los valores de los parámetros que hacen que el error del modelo sea lo más pequeño posible.


# Mostrar las fuentes utilizadas

In [12]:
print("=" * 70)
print("DOCUMENTOS UTILIZADOS")
print("=" * 70)

for i, doc in enumerate(result["retrieved_documents"]):

    print(f"\nDOCUMENTO {i+1}")
    print("-" * 40)

    print(doc[:500])

DOCUMENTOS UTILIZADOS

DOCUMENTO 1
----------------------------------------
v. convexidad y su importancia
se discuti ó que la p érdida cuadr ática en regresi ón lineal
induce una superficie convexa respecto a par ámetros, por lo
que el m ´ınimo local coincide con el m ´ınimo global. esta
propiedad da estabilidad al entrenamiento.
vi. descenso degradiente
la actualizaci ón de par ámetros vista en diapositivas fue:
w←w−α ∂l
∂w (7)
b←b−α ∂l
∂b (8)
conαcomo hiperpar ámetro de aprendizaje (learning rate).
vi-a. interpretaci ón del learning rate
αpeque ño: pasos cortos y con

DOCUMENTO 2
----------------------------------------
muestra, mientras que las dependientes son los valores a
predecir y es afectada por las variables independientes.
vi. funciones convexas y no convexas
las funciones convexas tienen la propiedad anal´ıtica de con-
tar con un único m´ınimo global. mientras que las funciones no
convexas pueden presentar m últiples m´ınimos locales adem ás
del m´ınimo global.
figura 1. e

# Mostrar Metadatos

In [13]:
print("=" * 70)
print("METADATOS")
print("=" * 70)

for i, metadata in enumerate(result["metadatas"]):

    print(f"\nFUENTE {i+1}")

    for key, value in metadata.items():
        print(f"{key}: {value}")

METADATOS

FUENTE 1
nombre_archivo: 4_SEMANA_AI_20260310_2.pdf
semana: 4
tema_principal: I. RESUMENGENERAL DE LASESI ´ON
total_chunks: 5
secciones: I. RESUMENGENERAL DE LASESI ´ON, II. REPASO DEK-NEARESTNEIGHBORS(KNN), III. MODELOESTAD ´ISTICO YREGRESI ´ONLINEAL, IV. FUNCI ´ON DEP ´ERDIDA: MSEY COMPARACI ´ON CON, V. CONVEXIDAD Y SU IMPORTANCIA
autor: Allan Bolaños Barrientos
chunk_numero: 4
fecha: 2026-03-10

FUENTE 2
autor: Steven Sequeira Araya
total_chunks: 7
nombre_archivo: 4_SEMANA_AI_20260310_1.pdf
semana: 4
fecha: 2026-03-10
secciones: I. REPASO DE K-NEAREST NEIGHBORS, II. MODELO ESTAD ´ISTICO, III. REGRESI ´ON LINEAL, IV. IDEA PRINCIPAL, V. MIN SQUARE ERROR (MSE)
tema_principal: I. REPASO DE K-NEAREST NEIGHBORS
chunk_numero: 4

FUENTE 3
autor: Kendell Garbanzo Calvo
fecha: 2026-03-17
chunk_numero: 1
total_chunks: 8
nombre_archivo: 5_SEMANA_AI_20260317_1.pdf
semana: 5
secciones: I. REPASO DE LA CLASE ANTERIOR, II. SESGO YVARIANZA, III. TRAININGSET, IV. TESTINGSET, V. VALIDATIONS